In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

bronze_df = spark.table(
    "customer360_dev.bronze.addresses"
)

customers_df = spark.table(
    "customer360_dev.silver.customers"
).select("customer_id")

In [0]:
typed_df = (
    bronze_df
    .withColumn(
        "effective_from_cast",
        F.to_timestamp("effective_from")
    )
    .withColumn(
        "effective_to_cast",
        F.to_timestamp("effective_to")
    )
    .withColumn(
        "created_at_cast",
        F.to_timestamp("created_at")
    )
    .withColumn(
        "updated_at_cast",
        F.to_timestamp("updated_at")
    )
    .withColumn(
        "address_type_clean",
        F.upper(F.trim("address_type"))
    )
    .withColumn(
        "city_clean",
        F.initcap(F.trim("city"))
    )
    .withColumn(
        "state_clean",
        F.initcap(F.trim("state"))
    )
    .withColumn(
        "country_clean",
        F.upper(F.trim("country"))
    )
)

Add referential check against Silver Customers

In [0]:
validated_df = (
    typed_df
    .join(
        customers_df.withColumn(
            "_customer_exists",
            F.lit(True)
        ),
        on="customer_id",
        how="left"
    )
)

Define data-quality rules

In [0]:
validated_df = (
    validated_df
    .withColumn(
        "dq_missing_address_id",
        F.col("address_id").isNull()
        | (F.trim(F.col("address_id")) == "")
    )
    .withColumn(
        "dq_missing_customer_id",
        F.col("customer_id").isNull()
        | (F.trim(F.col("customer_id")) == "")
    )
    .withColumn(
        "dq_customer_not_found",
        F.col("_customer_exists").isNull()
    )
    .withColumn(
        "dq_invalid_address_type",
        ~F.col("address_type_clean").isin(
            "HOME",
            "BILLING",
            "SHIPPING"
        )
    )
    .withColumn(
        "dq_invalid_effective_dates",
        F.col("effective_from_cast").isNull()
        | (
            F.col("effective_to_cast").isNotNull()
            & (
                F.col("effective_to_cast")
                < F.col("effective_from_cast")
            )
        )
    )
)

In [0]:
validated_df = validated_df.withColumn(
    "dq_critical_failure",
    F.col("dq_missing_address_id")
    | F.col("dq_missing_customer_id")
    | F.col("dq_customer_not_found")
    | F.col("dq_invalid_address_type")
    | F.col("dq_invalid_effective_dates")
)

Split valid and rejected rows

In [0]:
reject_df = validated_df.filter(
    F.col("dq_critical_failure")
)

valid_df = validated_df.filter(
    ~F.col("dq_critical_failure")
)

Deduplicate addresses deterministically

In [0]:
address_window = (
    Window
    .partitionBy("address_id")
    .orderBy(
        F.col("updated_at_cast").desc_nulls_last(),
        F.col("_ingestion_timestamp").desc()
    )
)

deduped_df = (
    valid_df
    .withColumn(
        "_row_number",
        F.row_number().over(address_window)
    )
    .filter(F.col("_row_number") == 1)
)

Validate primary-address behavior

CUST1001
HOME
is_primary = true

CUST1001
HOME
is_primary = true%md


In [0]:
primary_conflicts_df = (
    deduped_df
    .filter(
        (F.col("is_primary") == "true")
        & (
            F.col("effective_to_cast").isNull()
        )
    )
    .groupBy(
        "customer_id",
        "address_type_clean"
    )
    .count()
    .filter(
        F.col("count") > 1
    )
)

Build final Silver structure

In [0]:
silver_df = deduped_df.select(
    "address_id",
    "customer_id",

    F.col("address_type_clean").alias(
        "address_type"
    ),

    "address_line1",
    "address_line2",

    F.col("city_clean").alias("city"),
    F.col("state_clean").alias("state"),
    "postal_code",

    F.col("country_clean").alias("country"),

    F.col("is_primary").cast("boolean").alias(
        "is_primary"
    ),

    F.col("effective_from_cast").alias(
        "effective_from"
    ),
    F.col("effective_to_cast").alias(
        "effective_to"
    ),

    F.col("is_current").cast("boolean").alias(
        "is_current"
    ),

    F.col("created_at_cast").alias("created_at"),
    F.col("updated_at_cast").alias("updated_at"),

    "_source_file",
    "_source_system",
    "_load_type",
    "_ingestion_timestamp",

    F.current_timestamp().alias(
        "_silver_processed_at"
    )
)

In [0]:
(
    silver_df
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(
        "customer360_dev.silver.addresses"
    )
)

Write rejected addresses

In [0]:
quarantine_df = (
    reject_df
    .withColumn(
        "rejection_reason",
        F.concat_ws(
            "; ",
            F.when(
                F.col("dq_missing_address_id"),
                F.lit("MISSING_ADDRESS_ID")
            ),
            F.when(
                F.col("dq_missing_customer_id"),
                F.lit("MISSING_CUSTOMER_ID")
            ),
            F.when(
                F.col("dq_customer_not_found"),
                F.lit("CUSTOMER_NOT_FOUND")
            ),
            F.when(
                F.col("dq_invalid_address_type"),
                F.lit("INVALID_ADDRESS_TYPE")
            ),
            F.when(
                F.col("dq_invalid_effective_dates"),
                F.lit("INVALID_EFFECTIVE_DATES")
            )
        )
    )
    .withColumn(
        "_quarantined_at",
        F.current_timestamp()
    )
)

In [0]:
if "_pipeline_run_id" not in spark.table("customer360_dev.quarantine.address_rejects").columns:
    spark.sql("ALTER TABLE customer360_dev.quarantine.address_rejects ADD COLUMNS (_pipeline_run_id STRING)")
(
    quarantine_df
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(
        "customer360_dev.quarantine.address_rejects"
    )
)

Verify

In [0]:
bronze_count = spark.table(
    "customer360_dev.bronze.addresses"
).count()

silver_count = spark.table(
    "customer360_dev.silver.addresses"
).count()

reject_count = spark.table(
    "customer360_dev.quarantine.address_rejects"
).count()

print(f"Bronze : {bronze_count:,}")
print(f"Silver : {silver_count:,}")
print(f"Reject : {reject_count:,}")

Bronze : 1,200,078
Silver : 1,178,400
Reject : 21,678
